In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle

In [ ]:
## Load trained model, scaler pickle file and OHE
model = load_model('model.h5')

#load the encoder and scaler
with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [ ]:
# OHE for 'Geography'
input_data = {'Geography': 'France'}
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df


In [ ]:
## Combine OHE with input data
input_df = pd.concat([pd.DataFrame([input_data]), geo_encoded_df], axis=1)
## Combine OHE with input data
input_df = pd.concat([pd.DataFrame([input_data]), geo_encoded_df], axis=1)


In [ ]:
#encode categorical variables
if 'Gender' not in input_df.columns:
    input_df['Gender'] = input_data.get('Gender', 'Male')

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df
input_df

In [ ]:
## concatenate the OHE columns with the input data
input_df = pd.concat([input_df.drop("Geography", axis =1), geo_encoded_df], axis=1)
input_df

In [ ]:
#Scaling the input data
# prepare input_df: remove duplicate columns and align to scaler's expected features
input_df = input_df.loc[:, ~input_df.columns.duplicated()]

if hasattr(scaler, "feature_names_in_"):
    expected = list(scaler.feature_names_in_)
    # add any missing features with zeros and reorder to match training
    prepared = input_df.reindex(columns=expected, fill_value=0)
else:
    # fallback: assume scaler was fitted on numeric array and just use existing columns
    prepared = input_df

# ensure numeric dtype
prepared = prepared.astype(float)

input_scaled = scaler.transform(prepared)
input_scaled

In [ ]:
#predict using the trained model
prediction = model.predict(input_scaled)
prediction

In [ ]:
prediction_probability = prediction[0][0]
prediction_probability

In [ ]:
if prediction_probability >= 0.5:
    print("The model predicts that the customer will leave with a probability of {:.2f}%".format(prediction_probability * 100))
else:
    print("The model predicts that the customer will stay with a probability of {:.2f}%".format((1 - prediction_probability) * 100))